In [1]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
import gradio as gr

c:\Users\user\Documents\Market_Campaign_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = "openai/gpt-oss-20b:free"
system_prompt = """
You are a message router.

Your task is to classify the user message.

Example routes:

1. greeting:
- Hello
- Hi
- Thanks
- How are you

2. marketing:
- Creating campaigns
- Product promotion
- Audience analysis
- Marketing strategy


Return only JSON.



{
 "route": "greeting"
}

or

{
 "route": "marketing"
}
"""


In [3]:
import json
client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


async def router(user_message):

    response = await client.chat.completions.create(
        model=model,

        messages=[
            {
                "role":"system",
                "content":system_prompt
            },
            {
                "role":"user",
                "content":user_message
            }
        ],

        response_format={
            "type":"json_object"
        }
    )


    result = json.loads(
        response.choices[0].message.content
    )

    return result["route"]

In [4]:
await router("I want marketing plan")

'marketing'

In [5]:
async def make_subtasks(task):
        response = await client.chat.completions.create(
        model=model,
        
        messages=[
            {
                "role":"system",
                "content":"Split Markting task into 3 subtasks"
            },
            {
                "role":"user",
                "content":task
            }
        ],
    )

        steps = [s.strip(" .") for s in response.choices[0].message.content.split("\n") if s.strip()]
        return steps[:3]

In [6]:
async def decision_making(user_message):

    route = await router(user_message)


    if route == "greeting":

        return "Hello! How can I help you today?"


    elif route == "marketing":

        return make_subtasks(user_message)

In [7]:
async def Agent_Experts(role, task):
    role_prompts = {
    "Researcher": "You are a Market researcher. Analyze audience needs.",
    "Copywriter": "You are a copywriter. Write catchy ad copy.",
    "Planner": "You are a campaign planner. Suggest a weekly schedule."
}
    response = await client.chat.completions.create(
        model=model,
        
        messages=[
            {
                "role":"system",
                "content":role_prompts[role]
            },
            {
                "role":"user",
                "content":task
            }
        ],
        temperature=0.6
    )

    return f"--- {role} ---\n{response.choices[0].message.content}"

In [8]:
import asyncio


async def paralize(task):
    roles = ["Researcher", "Copywriter", "Planner"]
    tasks = [Agent_Experts(r, task) for  r in roles]
    results = await asyncio.gather(*tasks)
    return "\n\n".join(results)


In [9]:
async def review_and_optimize(text: str):
    review = await client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":"system",
                "content":"Review if this marketing content is clear, persuasive and actionable. reply 'yes' or 'no' and give feedback." 
            },
            {
                "role":"user",
                "content":text
            }
        ],
        temperature=0
    )

    feedback = review.choices[0].message.content.strip().lower()

    if "yes" in feedback:
        return text + "\n\n✅ Approved by Marketing Reviwer"

    else:
        improved = await client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":"system",
                "content":f"Improve the marketing content based on this feddback: {feedback}" 
            },
            {
                "role":"user",
                "content":text
            }
        ],
        temperature=0.5
        )
        return improved.choices[0].message.content + "\n\n Improved after reviwe"
    

In [10]:
async def Agent_Workflow(user_input):

    route = await router(user_input)

    if not route:
        return "If you have any question, tell me!"


    if route == "greeting":
        return "Hello! How can I help you today?"


    elif route == "marketing":

        subtasks = await make_subtasks(user_input)

        parallel_results = await paralize(
            " | ".join(subtasks)
        )

        final = await review_and_optimize(
            parallel_results
        )

        return final

In [11]:
async def chat(message, history):
    response = await Agent_Workflow(message)
    return response


demo = gr.ChatInterface(
    fn=chat,
    title="Marketing Campaign AI Agent",
    description="An AI Agent that creates marketing campaigns."
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
